# Notebook 02: Feature Extraction — MELD

Extracts GoEmotions distributions, computes per-anchor conditional AMD,
and derives rapport proxies (inter-speaker emotion agreement) from the
MELD (Multimodal EmotionLines Dataset) corpus.

Outputs:
- `meld_features.parquet`: per-utterance features
- `meld_conversation_summary.parquet`: per-dialogue AMD and rapport metrics

In [ ]:
!pip install -q transformers torch pandas numpy tqdm scikit-learn pyarrow

In [ ]:
import json
import os
import re
import warnings
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm import tqdm
from transformers import pipeline

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

## 1. Load MELD Dataset

In [ ]:
!mkdir -p meld_data
!wget -q -O meld_data/train_sent_emo.csv "https://raw.githubusercontent.com/declare-lab/MELD/master/data/MELD/train_sent_emo.csv"
!wget -q -O meld_data/dev_sent_emo.csv "https://raw.githubusercontent.com/declare-lab/MELD/master/data/MELD/dev_sent_emo.csv"
!wget -q -O meld_data/test_sent_emo.csv "https://raw.githubusercontent.com/declare-lab/MELD/master/data/MELD/test_sent_emo.csv"

train_df = pd.read_csv("meld_data/train_sent_emo.csv")
dev_df = pd.read_csv("meld_data/dev_sent_emo.csv")
test_df = pd.read_csv("meld_data/test_sent_emo.csv")

meld_df = pd.concat([train_df, dev_df, test_df], ignore_index=True)
print(f"Total utterances: {len(meld_df)}")
print(f"Columns: {list(meld_df.columns)}")
print(f"\nEmotion distribution:")
print(meld_df["Emotion"].value_counts())
print(f"\nSentiment distribution:")
print(meld_df["Sentiment"].value_counts())
print(f"\nUnique dialogues: {meld_df['Dialogue_ID'].nunique()}")
print(f"Unique speakers: {meld_df['Speaker'].nunique()}")

In [ ]:
speaker_counts = meld_df.groupby("Dialogue_ID")["Speaker"].nunique()
dyadic_ids = speaker_counts[speaker_counts == 2].index

usable_ids = speaker_counts[speaker_counts >= 2].index
meld_filtered = meld_df[meld_df["Dialogue_ID"].isin(usable_ids)].copy()

print(f"Strictly dyadic dialogues: {len(dyadic_ids)}")
print(f"Dialogues with >=2 speakers: {len(usable_ids)}")
print(f"Utterances after filtering: {len(meld_filtered)}")

def get_top2_speakers(group):
    top2 = group["Speaker"].value_counts().head(2).index.tolist()
    return group[group["Speaker"].isin(top2)]

meld_dyadic = meld_filtered.groupby("Dialogue_ID", group_keys=False).apply(get_top2_speakers)
meld_dyadic = meld_dyadic.sort_values(["Dialogue_ID", "Utterance_ID"]).reset_index(drop=True)

print(f"\nAfter restricting to top-2 speakers per dialogue:")
print(f"  Utterances: {len(meld_dyadic)}")
print(f"  Dialogues: {meld_dyadic['Dialogue_ID'].nunique()}")
print(f"\nMin turns per dialogue: {meld_dyadic.groupby('Dialogue_ID').size().min()}")
print(f"Median turns per dialogue: {meld_dyadic.groupby('Dialogue_ID').size().median():.0f}")
print(f"Max turns per dialogue: {meld_dyadic.groupby('Dialogue_ID').size().max()}")

## 2. Extract GoEmotions Distributions

In [ ]:
GOEMOTIONS_LABELS = [
    "admiration", "amusement", "anger", "annoyance", "approval",
    "caring", "confusion", "curiosity", "desire", "disappointment",
    "disapproval", "disgust", "embarrassment", "excitement", "fear",
    "gratitude", "grief", "joy", "love", "nervousness",
    "optimism", "pride", "realization", "relief", "remorse",
    "sadness", "surprise", "neutral",
]

goemotions_pipeline = pipeline(
    "text-classification",
    model="SamLowe/roberta-base-go_emotions",
    top_k=None,
    device=0 if DEVICE == "cuda" else -1,
    truncation=True,
    max_length=512,
)
print("GoEmotions pipeline loaded")

In [ ]:
BATCH_SIZE = 32

texts = meld_dyadic["Utterance"].tolist()
texts_clean = [str(t).strip() if pd.notna(t) and str(t).strip() else "[empty]" for t in texts]

all_emotion_dists = []

for i in tqdm(range(0, len(texts_clean), BATCH_SIZE), desc="GoEmotions extraction"):
    batch = texts_clean[i:i + BATCH_SIZE]
    results = goemotions_pipeline(batch)

    for result in results:
        score_map = {item["label"]: item["score"] for item in result}
        dist = np.array([score_map.get(label, 0.0) for label in GOEMOTIONS_LABELS])
        dist = dist / (dist.sum() + 1e-10)
        all_emotion_dists.append(dist)

meld_dyadic["goemotions_dist"] = all_emotion_dists
print(f"Extracted GoEmotions for {len(all_emotion_dists)} utterances")

## 3. Topic Clustering and Context Assignment

In [ ]:
TOPIC_K = 5

all_records = []
convo_grouped = defaultdict(list)

for dialog_id, group in tqdm(meld_dyadic.groupby("Dialogue_ID"), desc="Preparing records"):
    group = group.sort_values("Utterance_ID")
    speakers = group["Speaker"].unique().tolist()

    for turn_idx, (_, row) in enumerate(group.iterrows()):
        record = {
            "dialog_id": str(dialog_id),
            "turn_idx": turn_idx,
            "speaker": row["Speaker"],
            "text": str(row["Utterance"]) if pd.notna(row["Utterance"]) else "",
            "emotion": row["Emotion"],
            "sentiment": row["Sentiment"],
            "goemotions_dist": row["goemotions_dist"],
        }
        convo_grouped[str(dialog_id)].append(record)

for dialog_id, records in tqdm(convo_grouped.items(), desc="Topic clustering"):
    texts_in_convo = [r["text"] for r in records]
    effective_k = min(TOPIC_K, len(texts_in_convo))

    if effective_k < 2:
        for r in records:
            r["topic_cluster"] = 0
        continue

    vectorizer = TfidfVectorizer(max_features=1000, stop_words="english")
    tfidf_matrix = vectorizer.fit_transform(texts_in_convo)
    kmeans = KMeans(n_clusters=effective_k, random_state=RANDOM_SEED, n_init=10)
    clusters = kmeans.fit_predict(tfidf_matrix)

    for idx, r in enumerate(records):
        r["topic_cluster"] = int(clusters[idx])

for dialog_id, records in convo_grouped.items():
    for idx, r in enumerate(records):
        preceding_emo = records[idx - 1]["emotion"] if idx > 0 else "none"
        r["context"] = f"{preceding_emo}__topic_{r['topic_cluster']}"

print(f"Processed {len(convo_grouped)} dialogues")

## 4. Compute Conditional AMD Per Dialogue

In [ ]:
from nltk.corpus import stopwords
import nltk
nltk.download("stopwords", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

STOPWORDS = set(stopwords.words("english"))
WORD_PATTERN = re.compile(r"\b[a-z]{2,}\b")
MIN_ANCHOR_FREQ = 3
MIN_PER_CELL = 2  


def extract_content_words(text):
    words = WORD_PATTERN.findall(text.lower())
    return [w for w in words if w not in STOPWORDS]


def total_variation(p, q):
    return 0.5 * np.sum(np.abs(np.asarray(p) - np.asarray(q)))


print(f"Anchor config: min_freq={MIN_ANCHOR_FREQ}, min_per_cell={MIN_PER_CELL}")

In [ ]:
conversation_summaries = []

for dialog_id, records in tqdm(convo_grouped.items(), desc="Computing AMD per dialogue"):
    speakers = list(set(r["speaker"] for r in records))
    if len(speakers) < 2:
        continue

    spk1, spk2 = speakers[0], speakers[1]
    texts_spk1 = [r["text"] for r in records if r["speaker"] == spk1]
    texts_spk2 = [r["text"] for r in records if r["speaker"] == spk2]

    words_spk1, words_spk2 = [], []
    for t in texts_spk1:
        words_spk1.extend(extract_content_words(t))
    for t in texts_spk2:
        words_spk2.extend(extract_content_words(t))

    counter1, counter2 = Counter(words_spk1), Counter(words_spk2)
    shared = set(counter1.keys()) & set(counter2.keys())
    anchors = sorted([w for w in shared if counter1[w] + counter2[w] >= MIN_ANCHOR_FREQ])

    if not anchors:
        continue

    anchor_map = defaultdict(lambda: defaultdict(list))
    for r in records:
        text_words = set(WORD_PATTERN.findall(r["text"].lower()))
        for anchor in anchors:
            if anchor in text_words:
                anchor_map[(anchor, r["speaker"])][r["context"]].append(
                    np.array(r["goemotions_dist"])
                )

    d_marg_vals, d_cond_vals = [], []
    n_emo = len(GOEMOTIONS_LABELS)

    for anchor in anchors:
        g1 = anchor_map.get((anchor, spk1), {})
        g2 = anchor_map.get((anchor, spk2), {})

        ed1, ed2, cnt1, cnt2 = {}, {}, {}, {}
        for ctx, dists in g1.items():
            if len(dists) >= MIN_PER_CELL:
                ed1[ctx] = np.mean(dists, axis=0)
                cnt1[ctx] = len(dists)
        for ctx, dists in g2.items():
            if len(dists) >= MIN_PER_CELL:
                ed2[ctx] = np.mean(dists, axis=0)
                cnt2[ctx] = len(dists)

        if not ed1 or not ed2:
            continue

        t1 = sum(cnt1.values())
        t2 = sum(cnt2.values())
        cw1 = {c: n / t1 for c, n in cnt1.items()}
        cw2 = {c: n / t2 for c, n in cnt2.items()}

        all_ctx = set(cw1.keys()) | set(cw2.keys())
        marg1 = sum(cw1.get(c, 0) * ed1.get(c, np.zeros(n_emo)) for c in all_ctx)
        marg2 = sum(cw2.get(c, 0) * ed2.get(c, np.zeros(n_emo)) for c in all_ctx)
        d_marg_vals.append(total_variation(marg1, marg2))

        shared_ctx = set(ed1.keys()) & set(ed2.keys())
        if shared_ctx:
            wt_sum, tv_sum = 0.0, 0.0
            for c in shared_ctx:
                w = cnt1[c] + cnt2[c]
                tv_sum += w * total_variation(ed1[c], ed2[c])
                wt_sum += w
            if wt_sum > 0:
                d_cond_vals.append(tv_sum / wt_sum)

    if not d_marg_vals:
        continue

    emos_spk1 = [r["emotion"] for r in records if r["speaker"] == spk1]
    emos_spk2 = [r["emotion"] for r in records if r["speaker"] == spk2]
    min_len = min(len(emos_spk1), len(emos_spk2))
    if min_len > 0:
        agreements = sum(1 for i in range(min_len) if emos_spk1[i] == emos_spk2[i])
        emotion_agreement = agreements / min_len
    else:
        emotion_agreement = np.nan

    POSITIVE_EMOTIONS = {"joy", "surprise"}
    pos_spk1 = sum(1 for e in emos_spk1 if e in POSITIVE_EMOTIONS) / max(len(emos_spk1), 1)
    pos_spk2 = sum(1 for e in emos_spk2 if e in POSITIVE_EMOTIONS) / max(len(emos_spk2), 1)
    shared_positivity = (pos_spk1 + pos_spk2) / 2

    SENTIMENT_MAP = {"positive": 1.0, "neutral": 0.0, "negative": -1.0}
    valences = [SENTIMENT_MAP.get(r["sentiment"], 0.0) for r in records]
    mean_valence = np.mean(valences) if valences else 0.0

    conversation_summaries.append({
        "dialog_id": dialog_id,
        "n_turns": len(records),
        "n_anchors": len(anchors),
        "mean_d_marg": np.mean(d_marg_vals),
        "mean_d_cond": np.mean(d_cond_vals) if d_cond_vals else np.nan,
        "rapport_proxy": emotion_agreement,
        "shared_positivity": shared_positivity,
        "mean_valence": mean_valence,
    })

print(f"\nDialogues with valid anchors: {len(conversation_summaries)}")

## 5. Save Features to Parquet

In [ ]:
OUTPUT_DIR = Path("../data")
OUTPUT_DIR.mkdir(exist_ok=True)

flat_records = []
for dialog_id, records in convo_grouped.items():
    for r in records:
        flat = {
            "dialog_id": r["dialog_id"],
            "turn_idx": r["turn_idx"],
            "speaker": r["speaker"],
            "text": r["text"],
            "emotion": r["emotion"],
            "sentiment": r["sentiment"],
            "topic_cluster": r["topic_cluster"],
            "context": r["context"],
        }
        for emo_idx, emo_name in enumerate(GOEMOTIONS_LABELS):
            flat[f"ge_{emo_name}"] = r["goemotions_dist"][emo_idx]
        flat_records.append(flat)

utt_df = pd.DataFrame(flat_records)
utt_df.to_parquet(OUTPUT_DIR / "meld_features.parquet", index=False)
print(f"Utterance features saved: {utt_df.shape}")

summary_df = pd.DataFrame(conversation_summaries)
summary_df.to_parquet(OUTPUT_DIR / "meld_conversation_summary.parquet", index=False)
print(f"Conversation summaries saved: {summary_df.shape}")
print(f"\nSummary stats:")
print(summary_df[["mean_d_marg", "mean_d_cond", "rapport_proxy", "shared_positivity"]].describe())

## [Colab only] Save outputs to Google Drive
Run this cell to copy parquet files to Google Drive for download.
**Remove this cell before submitting.**

In [ ]:
# === [Colab only] Save to Google Drive — remove before submitting ===
from google.colab import drive
drive.mount("/content/drive")

import shutil

DRIVE_DEST = Path("/content/drive/MyDrive/phase-transition-amd/data")
DRIVE_DEST.mkdir(parents=True, exist_ok=True)

for f in ["meld_features.parquet", "meld_conversation_summary.parquet"]:
    src = OUTPUT_DIR / f
    if src.exists():
        shutil.copy2(src, DRIVE_DEST / f)
        print(f"Copied {f} -> {DRIVE_DEST}")

print(f"\nAll MELD outputs saved to Google Drive at: {DRIVE_DEST}")